In [1]:
import os
import sys
import yaml
import shutil
import pandas as pd
import pyemu
from pathlib import Path

def prepare_MOU_files(objectives='model_misfit', scenario_name=None, param_set_selector=None):
    """
    Build template folder and PST. param_set_selector can be:
      - None -> use defaults in proj6.yml
      - int -> 0-based index into param_sets
      - str -> name of param_set to select
    """
    # 1. Setup Absolute Paths
    root = Path(r"C:\Python\Personal\proj6\codes").resolve()
    pycap_run_name = "proj6"
    python_exe = sys.executable  # Path to python.exe

    if scenario_name is None:
        scenario_name = objectives

    parent_run_path = root / "runs"
    base_run_path = parent_run_path / "base"
    pest_path = parent_run_path / "pest"
    template_path = pest_path / f"{scenario_name}_template"
    script_source = root / "scripts" / "run_theis_forward.py"

    # 2. Clean & Recreate Template Folder
    if template_path.exists():
        import time
        try:
            shutil.rmtree(template_path)
        except PermissionError:
            time.sleep(1)
            shutil.rmtree(template_path)
    template_path.mkdir(parents=True)

    # 3. Read base proj6.yml and (optionally) apply param_set
    with open(base_run_path / f"{pycap_run_name}.yml", 'r') as ifp:
        indat = yaml.safe_load(ifp)

    # apply param_set_selector if requested
    if param_set_selector is not None:
        ps = indat.get('param_sets', [])
        if isinstance(param_set_selector, int):
            if param_set_selector < 0 or param_set_selector >= len(ps):
                raise IndexError("param_set_selector index out of range")
            sel = ps[param_set_selector]
        else:
            matches = [p for p in ps if p.get('name') == str(param_set_selector)]
            if not matches:
                raise KeyError(f"No param_set named '{param_set_selector}'")
            sel = matches[0]
        # apply fields if present
        for k in ('T', 'S', 't_eval'):
            if k in sel:
                indat[k] = sel[k]

    # 4. Replace well Q values with template tokens (for PEST)
    well_keys = [i for i in indat.keys() if i.startswith('well_')]
    for k in well_keys:
        # token name like well_1__q (same as before)
        indat[k]['Q'] = f"~{k + '__q':^20}~"

    # 5. Write the .tpl template
    with open(template_path / f"{pycap_run_name}.yml.tpl", 'w') as ofp:
        ofp.write('ptf ~\n')
        yaml.dump(indat, ofp, default_flow_style=False, sort_keys=False)

    # 6. Copy the modified in-file (proj6.yml) into the template folder so model can read it
    with open(template_path / f"{pycap_run_name}.yml", 'w') as ofp:
        yaml.dump(indat, ofp, default_flow_style=False, sort_keys=False)

    # 7. Create Instruction File (INS) & Dummy Out
    with open(template_path / 'allobs.out.ins', 'w') as ofp:
        ofp.write('pif ~\n')
        for k in sorted(well_keys):
            ofp.write(f'l1 w !{k}!\n')

    with open(template_path / 'allobs.out', 'w') as ofp:
        for k in sorted(well_keys):
            ofp.write(f"{k} 0.0\n")

    # 8. Copy Script Locally
    shutil.copy(script_source, template_path / "run_theis_forward.py")

    # 9. Build the PST Object
    cwd = os.getcwd()
    os.chdir(template_path)

    pst = pyemu.Pst.from_io_files(
        tpl_files=[f"{pycap_run_name}.yml.tpl"],
        in_files=[f"{pycap_run_name}.yml"],
        ins_files=["allobs.out.ins"],
        out_files=["allobs.out"]
    )

    # ensure correct model command (use the copied script name)
    pst.model_command = [f"{python_exe} run_theis_forward.py"]

    # MOU Required Settings
    pst.parameter_data.loc[:, "partrans"] = "none"
    pst.parameter_data.loc[:, "parlbnd"] = 0.1
    pst.observation_data.loc[:, "obgnme"] = "less_misfit"
    pst.observation_data.loc[:, "weight"] = 1.0

    # Test Mode -1 (Check inputs and run once)
    pst.control_data.noptmax = -1

    pst.write(f"{scenario_name}.pst")
    os.chdir(cwd)

    return scenario_name, str(template_path)

In [2]:
name, path = prepare_MOU_files(scenario_name='baseline', param_set_selector="Low T, Low S")

# Run the test
import subprocess
pest_exe = r"C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe"
os.chdir(path)
subprocess.run([pest_exe, f"{name}.pst"], shell=True)

# CHECK THE RESULT
if os.path.exists("allobs.out"):
    with open("allobs.out", "r") as f:
        print("\n--- Model Output ---")
        print(f.read())
else:
    print("Error: allobs.out was not created.")


noptmax:-1, npar_adj:2, nnz_obs:2

--- Model Output ---
well_1 0.0
well_2 0.0

